---
title: "Wikidata Item Profile"
format:
  html:
    toc: false
    code-fold: true
    code-summary: "Show code"
execute:
  echo: false
  warning: false
  error: false
jupyter: python3
---

# Wikidata Item Profile

This notebook-backed page queries Wikidata with SPARQL and renders a styled HTML profile.

To profile another item, change the `item_id` value in Cell 3 and re-render the site.

In [1]:
from IPython.display import HTML, Markdown
from wikidata_profile import (
    build_statement_query,
    fetch_sparql_bindings,
    properties_from_bindings,
    render_profile_html,
)

In [2]:
item_id = "Q138572983"
item_id = item_id.strip().upper()

if not item_id.startswith("Q") or not item_id[1:].isdigit():
    raise ValueError("item_id must be a Wikidata Q-id like Q42 or Q138572983")

query = build_statement_query(item_id)
Markdown(
    f"## SPARQL Query Used for {item_id}\n```sparql\n" + query + "\n```"
 )

## SPARQL Query Used for Q138572983
```sparql
SELECT ?property ?propertyLabel ?value ?valueLabel WHERE {
  BIND(wd:Q138572983 AS ?item)
  ?item ?p ?statement .
  ?property wikibase:claim ?p .
  ?statement ?ps ?value .
  ?property wikibase:statementProperty ?ps .

  SERVICE wikibase:label { bd:serviceParam wikibase:language "[AUTO_LANGUAGE],en". }
}
ORDER BY ?propertyLabel
```

In [3]:
bindings = fetch_sparql_bindings(query)
properties = properties_from_bindings(bindings)
len(bindings)

9

In [4]:
HTML(render_profile_html(item_id, properties))

In [ ]:
import plotly.graph_objects as go
import math

# Create a graph from the properties
nodes_x = []
nodes_y = []
nodes_text = []
nodes_color = []
nodes_size = []

edges_x = []
edges_y = []

# Helper function to truncate long text
def truncate(text, length=20):
    return (text[:length] + '...') if len(str(text)) > length else str(text)

# Central node (main item)
center_x, center_y = 0, 0
nodes_x.append(center_x)
nodes_y.append(center_y)
nodes_text.append(item_id)
nodes_color.append("#FF6B6B")
nodes_size.append(30)

# Position properties in a circle around the center
num_properties = len(properties) if properties else 1
property_radius = 3

for i, prop in enumerate(properties):
    try:
        # Safely extract property label
        if isinstance(prop, dict):
            prop_label = prop.get("property_label") or prop.get("property_id") or f"P{i}"
        else:
            prop_label = f"Property {i}"
        
        prop_label = truncate(prop_label)
        
        # Calculate position on circle
        angle = (2 * math.pi * i) / max(num_properties, 1)
        prop_x = property_radius * math.cos(angle)
        prop_y = property_radius * math.sin(angle)
        
        nodes_x.append(prop_x)
        nodes_y.append(prop_y)
        nodes_text.append(prop_label)
        nodes_color.append("#4ECDC4")
        nodes_size.append(20)
        
        # Edge from center to property
        edges_x.extend([center_x, prop_x, None])
        edges_y.extend([center_y, prop_y, None])
        
        # Position objects around each property
        if isinstance(prop, dict) and "objects" in prop:
            objects = prop["objects"]
            if isinstance(objects, list):
                objects = objects[:5]  # Limit to first 5 objects
                num_objects = len(objects)
                object_radius = 1.5
                
                for j, obj in enumerate(objects):
                    try:
                        if isinstance(obj, dict):
                            obj_label = obj.get("label") or obj.get("value") or f"Object {j}"
                        else:
                            obj_label = str(obj)
                        
                        obj_label = truncate(obj_label, 15)
                        
                        # Calculate position around the property
                        obj_angle = (2 * math.pi * j) / max(num_objects, 1) + angle
                        obj_x = prop_x + object_radius * math.cos(obj_angle)
                        obj_y = prop_y + object_radius * math.sin(obj_angle)
                        
                        nodes_x.append(obj_x)
                        nodes_y.append(obj_y)
                        nodes_text.append(obj_label)
                        nodes_color.append("#95E1D3")
                        nodes_size.append(12)
                        
                        # Edge from property to object
                        edges_x.extend([prop_x, obj_x, None])
                        edges_y.extend([prop_y, obj_y, None])
                    except Exception as e:
                        print(f"Error processing object {j}: {e}")
                        
    except Exception as e:
        print(f"Error processing property {i}: {e}")

# Create figure
fig = go.Figure()

# Add edges
if edges_x:
    fig.add_trace(go.Scatter(
        x=edges_x, y=edges_y,
        mode='lines',
        line=dict(width=0.5, color='#888'),
        hoverinfo='none',
        showlegend=False
    ))

# Add nodes
fig.add_trace(go.Scatter(
    x=nodes_x, y=nodes_y,
    mode='markers+text',
    text=nodes_text,
    textposition="top center",
    hoverinfo='text',
    marker=dict(
        size=nodes_size,
        color=nodes_color,
        line=dict(width=2, color='white')
    ),
    showlegend=False
))

fig.update_layout(
    title=f"Wikidata Item Graph - {item_id}",
    showlegend=False,
    hovermode='closest',
    margin=dict(b=0, l=0, r=0, t=40),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=600
)

fig.show()

Der Befehl "pip" ist entweder falsch geschrieben oder
konnte nicht gefunden werden.


ModuleNotFoundError: No module named 'plotly'